In [91]:
# Filtering results based on average surfdiff score
import pandas as pd 
import csv 
import os
import glob
import numpy as np
import re

def read_seq_from_pdb(pdb_path, pdb_id = "structure", chain_id = "A"):
    """
    Reads the sequence from a PDB file.

    Args:
        pdb_path (str): Path to the PDB file.

    Returns:
        str: Amino acid sequence extracted from the PDB file.
    """

    from Bio.PDB import PDBParser
    from Bio.SeqUtils import seq1

    pdbparser = PDBParser()
    structure = pdbparser.get_structure(pdb_id, pdb_path)
    chains = {chain.id:seq1(''.join(residue.resname for residue in chain)) for chain in structure.get_chains()}

    return chains[chain_id]

def fasta_to_dict(fasta_file):
    """
    Reads a FASTA file and returns a dictionary where keys are headers (without '>') and values are sequences.

    Args:   
        fasta_file (str): The path to the FASTA file.

    Returns:
        dict: A dictionary with headers as keys and sequences as values.
    """
    fasta_dict = {}
    with open(fasta_file, 'r') as f:
        current_header = None
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                current_header = line[1:]
                fasta_dict[current_header] = ""
            elif current_header is not None:
                fasta_dict[current_header] += line
    return fasta_dict

# These residues are retrived from Table 3 of the following paper:
# Vo, Phillip, Daniel E. Connors, Brendan P. Sharkey, Stephen A. Gravina, Guy Servant, Chase T. McFarland, Tyler T. Pitkanen et al. 
# "Sweet Protein Allosteric Binding and Activation of the Human T1R2/R3 Sweet Receptor: A Simulation Model Validated by in Vitro Receptor Activation Assay."
# Biochemistry (2026).
# For Miraculin, the important histidine residues were chosen based on the following paper:
# Koizumi, Ayako, Asami Tsuchiya, Ken-ichiro Nakajima, Keisuke Ito, Tohru Terada, Akiko Shimizu-Ibuka, Loïc Briand, Tomiko Asakura, Takumi Misaka, and Keiko Abe. 
# "Human sweet taste receptor mediates acid-induced sweetness of miraculin."
# Proceedings of the National Academy of Sciences 108, no. 40 (2011): 16819-16824.
residues_to_check = {
    "Thaumatin-I_1LR2": [("K", 66), ("K", 77), ("K", 96), ("K", 105), ("K", 136), ("Y", 168), ("K", 186), ("D", 20)],
    "Thaumatin-II_3AOK": [("R", 66), ("K", 77), ("R", 81), ("K", 96), ("K", 105), ("K", 136), ("Y", 168), ("K", 186), ("D", 20)],
    "MNEI_1IV7": [('E', 22), ('Q', 27), ("C", 40), ("Y", 64), ("S", 66), ("D", 67), ("E", 1)],
    "Brazzein_4HE7": [("D", 1), ("K", 4), ("Y", 38), ("D", 39), ("E", 40), ("K", 41), ("R", 42), ("D", 49), ("Y", 53), ("D", 28), ("E", 35), ("R", 32)],
    "Honey_truffle_sweetener": [("R", 19), ("T", 20), ("K", 25), ("Q", 30), ("R", 56), ("Y", 61), ("Q", 63), ("R", 65), ("R", 74), ("S", 78), ("E", 88)],
    "Miraculin": [("H", 29), ("H", 59)]
}

def read_aligned_residues(tm_align_res_file):
    """
    Reads aligned residues from a TM-align result file.

    Args:
        tm_align_res_file (str): Path to the TM-align result file.

    Returns:
        dict: A dictionary where keys are positions in the query sequence and values are dictionaries containing the aligned residues from the query and target sequences.
    """
    aligned_residues = {}
    with open(tm_align_res_file, "r") as f:
        lines = f.readlines()
        query_seq = lines[19]
        align_info = lines[20]
        target_seq = lines[21]

    position = -1
    for index, residue in enumerate(query_seq):
        if query_seq[index] != "-":
             position += 1
        if align_info[index] == ":":
            aligned_residues[position] = {"query": residue, "target": target_seq[index]}
    
    return aligned_residues

repo_path = os.path.dirname(os.getcwd())
repo_path = repo_path.split("WP2_SweetProteinDiscovery")[0] + "WP2_SweetProteinDiscovery"
surfdiff_res = f"{repo_path}/6_structural_analyses/4_surfdiff_results"

# Read in the SurfDiff results for all models and extract the residue similarity scores
subdirs = os.listdir(surfdiff_res)
results = []
for sweet_protein in subdirs:
    res_path = os.path.join(surfdiff_res, sweet_protein, "tmp/output/scores")
    if sweet_protein == "Thaumatin-II_3AOK":
        res_path = os.path.join(surfdiff_res, sweet_protein, "batch_*", "tmp/output/scores")

    # List residue uniqueness score files and read in the residue similarity scores
    if not os.path.isdir(res_path):
        continue
    res_paths = glob.glob(res_path + "/*residue_uniqueness_scores.csv")
    print(sweet_protein, len(res_paths), "Files found")
    for res_file in res_paths:
        res_df = pd.read_csv(res_file, header = 0)

        model_name = os.path.basename(res_file)
        model_name = model_name.replace("_residue_uniqueness_scores.csv", "")
        model_name = model_name.split("___")[1] 

        # Add results
        if "residue_similarity_score" in res_df.columns:
            results.append([sweet_protein, model_name, res_df["residue_similarity_score"].tolist()])

# Make results into a dataframe
surfdiff_results_df = pd.DataFrame(results, columns = ["query", "target_id", "similarity_scores"])

# Read in the representative sequence df from earlier results
repseq_df = pd.read_csv(os.path.join(repo_path, "6_structural_analyses", "3_representative_structures", "representative_structures.csv"))
repseq_df["target_id"] = [i if "|" not in i else i.split("|")[1] for i in repseq_df["target"]]
all_repseq_df = repseq_df.copy()

# Loop over the dataframe and calculate mean residue uniqueness
repseq_df = repseq_df.merge(surfdiff_results_df, on = ["query", "target_id"])
repseq_df["avg_surfdiff_similarity"] = repseq_df["similarity_scores"].apply(np.mean)

# Calculate the average similarity score for the important residues for each model
for index, row in repseq_df.iterrows():
    important_residues = residues_to_check[row["query"]]
    important_residues = [i[1] for i in important_residues]
    important_res_scores = [row["similarity_scores"][i] for i in important_residues]
    repseq_df.loc[index, "avg_sweet_res_similarity"] = np.mean(important_res_scores)

repseq_df.sort_values(by=["query", "avg_sweet_res_similarity"], ascending = [True, False], inplace = True)
repseq_df

Brazzein_4HE7 26 Files found
Miraculin 888 Files found
Honey_truffle_sweetener 668 Files found
MNEI_1IV7 319 Files found


,query,target,evalue,bits,pident,fident,nident,qstart,qend,qlen,...,length_diff,interface_residues,grantham_score,pdb,cropped_tseq,cluster_id,target_id,similarity_scores,avg_surfdiff_similarity,avg_sweet_res_similarity
1306,Brazzein_4HE7,UniProt_tr|A0ABQ7DUU8|A0ABQ7DUU8_BRACR,7.610000e-03,184,29.3,0.293,17,1,54,54,...,0.092593,"[('D', 1, 'S'), ('K', 4, 'L'), ('Y', 38, 'W'),...",70.250000,Brazzein_4HE7___A0ABQ7DUU8,QDSECLKEYGGDVGFGFCAPRIYPSFCVQRCRADKGAISGKCIWGQ...,NaN,A0ABQ7DUU8,"[0.5270302755579208, 0.4653365866632601, 0.688...",0.554320,0.509341
225,Brazzein_4HE7,UniProt_tr|A0A8X8B8L9|A0A8X8B8L9_BRACI,2.981000e-02,163,28.8,0.288,17,1,53,54,...,0.148148,"[('D', 1, 'Q'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.500000,Brazzein_4HE7___A0A8X8B8L9,AQDSECLKEYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITG...,NaN,A0A8X8B8L9,"[0.3026833673365796, 0.1044607110940785, 0.441...",0.491869,0.490963
1308,Brazzein_4HE7,UniProt_tr|A0ABQ7DJF1|A0ABQ7DJF1_BRACR,2.453000e-02,166,30.5,0.305,18,1,53,54,...,0.111111,"[('D', 1, 'S'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.833333,Brazzein_4HE7___A0ABQ7DJF1,DSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITGDA...,NaN,A0ABQ7DJF1,"[0.3026833673365796, 0.1044607110940785, 0.439...",0.509846,0.477431
1433,Brazzein_4HE7,OneKP_scaffold-IPWB-2008469-Brassica_nigra,2.197000e-02,168,27.1,0.271,16,1,53,54,...,0.129630,"[('D', 1, 'S'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.833333,Brazzein_4HE7___OneKP_scaffold-IPWB-2008469-Br...,QDSECLREYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITGD...,NaN,OneKP_scaffold-IPWB-2008469-Brassica_nigra,"[0.5444435075269568, 0.4653365866632601, 0.577...",0.524192,0.470805
222,Brazzein_4HE7,UniProt_tr|A0A0D3BPK9|A0A0D3BPK9_BRAOL,2.453000e-02,166,28.8,0.288,17,1,53,54,...,0.148148,"[('D', 1, 'Q'), ('K', 4, 'L'), ('Y', 38, 'T'),...",72.500000,Brazzein_4HE7___A0A0D3BPK9,AQDSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITG...,NaN,A0A0D3BPK9,"[0.4633105527568497, 0.4461355449364914, 0.608...",0.510099,0.468148
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
845,Miraculin,UniProt_tr|M4EXH5|M4EXH5_BRACM,1.529000e-08,487,23.5,0.235,46,1,186,191,...,0.083770,"[('H', 29, 'A'), ('H', 59, 'R')]",57.500000,Miraculin___M4EXH5,SGEKEIVLDTIGRPVEFYAPYHAYFERPGANALICRYGSIETSGFS...,NaN,M4EXH5,"[0.1027522809026139, 0.2855078194145362, 0.491...",0.473611,0.046671
795,Miraculin,UniProt_tr|A0A9R1QYI6|A0A9R1QYI6_TRITD,2.273000e-09,524,21.4,0.214,45,1,190,191,...,0.020942,"[('H', 29, 'L'), ('H', 59, 'L')]",99.000000,Miraculin___A0A9R1QYI6,QAQPPPVYDIDGHELTGNNMYTIMAADRNLSDHCVSAGSFRREECN...,NaN,A0A9R1QYI6,"[0.1284403511282673, 0.3443261213357709, 0.634...",0.457263,0.034261
830,Miraculin,UniProt_tr|A0A9R1G5I8|A0A9R1G5I8_WHEAT,7.826000e-09,500,21.6,0.216,45,1,188,191,...,0.010471,"[('H', 29, 'L'), ('H', 59, 'F')]",99.500000,Miraculin___A0A9R1G5I8,QAQPPPVYDIDGHELTGNNMYTIMAADRNLSDHCISVGSFRSEECH...,NaN,A0A9R1G5I8,"[0.1284403511282673, 0.3446600593875392, 0.636...",0.448440,0.034261
831,Miraculin,UniProt_tr|A0A453GS48|A0A453GS48_AEGTS,7.826000e-09,500,21.6,0.216,45,1,188,191,...,0.010471,"[('H', 29, 'L'), ('H', 59, 'F')]",99.500000,Miraculin___A0A453GS48,QAQPPPVYDIDGHELTGNNMYTIMAADRNLSDHCISVGSFRSEECH...,NaN,A0A453GS48,"[0.1284403511282673, 0.3446355564613413, 0.638...",0.459073,0.034261


In [ ]:
repseq_df = repseq_df[['query', 'target_id', 'evalue', 'tm_score', 'RMSD', 'avg_pLDDT', 'model', 'source', 'length_diff', 'grantham_score', 'cropped_tseq', 'avg_surfdiff_similarity', 'avg_sweet_res_similarity']].copy()
repseq_df["seq_length"] = repseq_df["cropped_tseq"].apply(lambda x: len(x.replace("-", "")))
repseq_df["cysteine_count"] = repseq_df["cropped_tseq"].apply(lambda x: x.count("C"))

# Remove sequences with uneven number of cysteines
repseq_df = repseq_df[repseq_df["cysteine_count"] % 2 == 0]

# Remove sequences longer than 200 amino acids
repseq_df = repseq_df[repseq_df["seq_length"] <= 200]

# Remove sequences with cysteines, but for Brazzein, keep all sequneces
repseq_df = repseq_df[~((repseq_df["cysteine_count"] > 0) & (repseq_df["query"] != "Brazzein_4HE7"))]

# For each sweet protein, pick top 5 models based on avg_sweet_res_similarity
final_representatives = repseq_df.groupby("query").head(5)
final_representatives

,query,target_id,evalue,tm_score,RMSD,avg_pLDDT,model,source,length_diff,grantham_score,cropped_tseq,avg_surfdiff_similarity,avg_sweet_res_similarity,seq_length,cysteine_count
1306,Brazzein_4HE7,A0ABQ7DUU8,0.007610,0.82643,NaN,67.395254,OF3,UniProt,0.092593,70.250000,QDSECLKEYGGDVGFGFCAPRIYPSFCVQRCRADKGAISGKCIWGQ...,0.554320,0.509341,59,8
225,Brazzein_4HE7,A0A8X8B8L9,0.029810,0.77667,2.02,79.511935,AFDB,UniProt,0.148148,72.500000,AQDSECLKEYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITG...,0.491869,0.490963,62,8
1308,Brazzein_4HE7,A0ABQ7DJF1,0.024530,0.80621,NaN,84.091333,AF3,UniProt,0.111111,72.833333,DSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITGDA...,0.509846,0.477431,60,8
1433,Brazzein_4HE7,OneKP_scaffold-IPWB-2008469-Brassica_nigra,0.021970,0.81020,NaN,71.231311,OF3,OneKP,0.129630,72.833333,QDSECLREYGGNVGFGFCAPQTFPTICYIRCREDKDAKGGRCITGD...,0.524192,0.470805,61,8
222,Brazzein_4HE7,A0A0D3BPK9,0.024530,0.77567,1.99,80.220161,AFDB,UniProt,0.148148,72.500000,AQDSECLKEYGGNVGFGFCAPRTFPTICYIRCREDKDAKGGRCITG...,0.510099,0.468148,62,8
930,Honey_truffle_sweetener,D8RH23,0.000202,0.76785,2.38,92.812824,AFDB,UniProt,0.082645,80.454545,KLKGEDNYFALVIYYTEGPLTLYVANSWLGYGKWVRGKNDFTEIGS...,0.388768,0.357888,131,0
1537,Honey_truffle_sweetener,OneKP_scaffold-JBLI-2099947-Danaea_sp.,0.010580,0.73018,NaN,96.806905,AF3,OneKP,0.041322,75.454545,RGHAQWAAFHITYSGEGNLTLRNSRLWWGKWFDYPRQLTEIGGPDG...,0.357660,0.351448,126,0
1442,Honey_truffle_sweetener,OneKP_scaffold-ZZOL-2050369-Selaginella_staunt...,0.000426,0.77274,NaN,95.017752,AF3,OneKP,0.066116,80.454545,KGENNYFALVIYYTEGPLTLYVANSWLGYGKWVRGKNDFTEIGSPN...,0.388881,0.345915,129,0
1067,Honey_truffle_sweetener,A0A0D2SAQ9,0.005289,0.78123,2.42,92.741667,AFDB,UniProt,0.190083,72.818182,NAQQTTNGKLWNSSSEAITLTDKKVWQGSHYADFPEIIEDGDAREF...,0.368885,0.345740,144,0
1459,Honey_truffle_sweetener,OneKP_scaffold-OFWX-2012124-Mollugo_verticilla...,0.001726,0.77907,NaN,87.718231,OF3,OneKP,0.214876,90.818182,GNGVSALHVFYNASGAPLTFIDTYDYHGHGYSVPPPSDVQNGQWGA...,0.424348,0.344307,147,0
